In [27]:
# ============================================================
# CELL 1 — SETUP AND LOAD
# Notebook: 06_objective2_personalization.ipynb
#
# Loads:
#   wesad_features_selected.csv  (883 windows × 22 features)
#   best_model_obj1.pkl          (trained XGBoost)
#   scaler_obj1.pkl              (fitted StandardScaler)
#   feature_cols_obj1.pkl        (22 feature names)
# ============================================================

import numpy as np
import pandas as pd
import os
import sys
import warnings
import joblib
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb

COMPONENT_PATH = r"C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals"
sys.path.insert(0, os.path.join(COMPONENT_PATH, "src"))
from utils.config import RESULTS_PATH

# ── Load feature matrix ──────────────────────────────────────
csv_path    = os.path.join(
    RESULTS_PATH, "objective1",
    "wesad_features_selected.csv"
)
features_df = pd.read_csv(csv_path)

feature_cols = [c for c in features_df.columns
                if c not in ['label', 'subject_id']]

X        = features_df[feature_cols].values
y        = features_df['label'].values
subjects = features_df['subject_id'].values

# ── Load saved Objective 1 artifacts ─────────────────────────
models_path = os.path.join(COMPONENT_PATH, "saved_models")

best_model   = joblib.load(
    os.path.join(models_path, "best_model_obj1.pkl")
)
scaler_obj1  = joblib.load(
    os.path.join(models_path, "scaler_obj1.pkl")
)
saved_feats  = joblib.load(
    os.path.join(models_path, "feature_cols_obj1.pkl")
)

print("=" * 65)
print("OBJECTIVE 2 — PERSONALIZED ADAPTIVE MODEL")
print("=" * 65)
print(f"Feature matrix: {features_df.shape}")
print(f"Features: {len(feature_cols)}")
print(f"Subjects: {sorted(np.unique(subjects))}")
print()
print("LABEL DISTRIBUTION:")
for lid, name in {
    0:'Normal', 1:'Pre-stress', 2:'Stress'
}.items():
    count = (y == lid).sum()
    pct   = count / len(y) * 100
    print(f"  Class {lid} ({name:10s}): "
          f"{count:4d} ({pct:.1f}%)")
print()
print("OBJECTIVE 1 BASELINE (to compare against):")
print(f"  F1-macro:      0.617")
print(f"  F1-Pre-stress: 0.286")
print(f"  Ordinal MAE:   0.357")
print()
print("OBJECTIVE 2 TARGETS:")
print(f"  F1-Pre-stress improvement: > 5% over baseline")
print(f"  Target F1-Pre-stress: > 0.336")
print(f"  Target F1-macro: > 0.65")
print()
print("Cell 1 complete")

OBJECTIVE 2 — PERSONALIZED ADAPTIVE MODEL
Feature matrix: (883, 24)
Features: 22
Subjects: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]

LABEL DISTRIBUTION:
  Class 0 (Normal    ):  587 (66.5%)
  Class 1 (Pre-stress):   29 (3.3%)
  Class 2 (Stress    ):  267 (30.2%)

OBJECTIVE 1 BASELINE (to compare against):
  F1-macro:      0.617
  F1-Pre-stress: 0.286
  Ordinal MAE:   0.357

OBJECTIVE 2 TARGETS:
  F1-Pre-stress improvement: > 5% over baseline
  Target F1-Pre-stress: > 0.336
  Target F1-macro: > 0.65

Cell 1 complete


In [28]:
# ============================================================
# CELL 2 — LAYER 1: PERSONALIZED BASELINE CALIBRATION
#
# What it does:
#   Step 1: Take first 2 minutes of a person's session
#           as their personal calm baseline
#   Step 2: Calculate mean and std of each feature
#           during those 2 minutes
#   Step 3: Normalize ALL windows for that person
#           using their personal statistics
#           normalized = (value - personal_mean) / personal_std
#
# Why 2 minutes:
#   60-second window with 30-second step
#   2 minutes gives 3 overlapping windows
#   Average of 3 windows is stable
#   Paper 16: 2-minute baseline sufficient for HRV
#
# Example:
#   Person A baseline AVNN = 820ms  std = 45ms
#   Person B baseline AVNN = 650ms  std = 30ms
#
#   Person A stress AVNN = 720ms
#   Normalized: (720-820)/45 = -2.22  "very below baseline"
#
#   Person B stress AVNN = 600ms
#   Normalized: (600-650)/30 = -1.67  "below baseline"
#
#   Both show stress as deviation from their own normal
#   even though absolute values are very different

# Problem: z-score normalization with tiny std (0.110)
#          creates extreme values (-19, -20)
#          destroying model predictions
#
# Fix 1: Minimum std = 10% of mean (prevents tiny std)
# Fix 2: EDA features normalized by ratio (value/baseline)
#         instead of z-score
#         ratio=1.0 means at baseline
#         ratio=2.0 means twice baseline (stress)
# Fix 3: HRV features use z-score (they have larger std)
# ============================================================

# Which features are HRV vs EDA vs ACC
HRV_FEATURES = [
    'AVNN', 'MeanHR', 'SDHR', 'pNN20', 'SD1'
]
EDA_FEATURES = [
    'SCL_mean', 'SCL_slope', 'SCL_std',
    'EDA_mean', 'EDA_std', 'EDA_range', 'EDA_slope',
    'SCR_count', 'SCR_amp_mean', 'SCR_width_mean',
    'EDA_q10', 'EDA_q25', 'EDA_q75', 'EDA_q90',
    'deriv1_mean', 'deriv1_std', 'deriv2_mean',
    'wavelet_approx_mean', 'wavelet_approx_energy',
    'wavelet_d1_energy', 'wavelet_d2_energy',
    'wavelet_d3_energy'
]
ACC_FEATURES = ['ACC_mean', 'ACC_std', 'ACC_range']


def compute_personal_baseline(X_person, y_person,
                               n_baseline_windows=3):
    """
    Compute personal baseline from first N Normal windows.
    Fixed: minimum std = 10% of abs(mean) to prevent extremes.
    """
    normal_mask    = (y_person == 0)
    normal_windows = X_person[normal_mask]

    if len(normal_windows) == 0:
        baseline_mean = np.mean(X_person, axis=0)
        baseline_std  = np.std(X_person, axis=0)
    elif len(normal_windows) < n_baseline_windows:
        baseline_windows = normal_windows
        baseline_mean    = np.mean(baseline_windows, axis=0)
        baseline_std     = np.std(baseline_windows,  axis=0)
    else:
        baseline_windows = normal_windows[:n_baseline_windows]
        baseline_mean    = np.mean(baseline_windows, axis=0)
        baseline_std     = np.std(baseline_windows,  axis=0)

    # Fix: minimum std = 10% of absolute mean
    # Prevents tiny std creating extreme normalized values
    min_std = np.abs(baseline_mean) * 0.10
    min_std = np.where(min_std < 0.01, 0.01, min_std)
    baseline_std = np.where(
        baseline_std < min_std,
        min_std,
        baseline_std
    )

    return baseline_mean, baseline_std


def apply_personal_normalization(X_person, baseline_mean,
                                  baseline_std,
                                  feature_cols_list):
    """
    Robust normalization:
    HRV features: z-score (value - mean) / std
    EDA features: ratio   value / mean
    ACC features: z-score

    EDA ratio normalization:
      1.0 = exactly at your baseline level
      1.5 = 50% above your baseline (stress building)
      2.0 = double your baseline (clear stress)
      0.5 = half your baseline (very relaxed)

    This is more stable than z-score for EDA because:
      EDA values are always positive
      Ratios are bounded and interpretable
      Tiny std does not cause extreme values
    """
    X_norm = X_person.copy().astype(np.float64)

    for i, feat in enumerate(feature_cols_list):
        val  = X_person[:, i]
        mean = baseline_mean[i]
        std  = baseline_std[i]

        if feat in EDA_FEATURES:
            # Ratio normalization for EDA
            if abs(mean) > 0.001:
                X_norm[:, i] = val / mean
            else:
                X_norm[:, i] = val - mean
        else:
            # Z-score for HRV and ACC
            X_norm[:, i] = (val - mean) / std

    return X_norm


# ── Test fixed normalization on S2 ───────────────────────────
print("=" * 65)
print("LAYER 1 FIXED — ROBUST NORMALIZATION TEST ON S2")
print("=" * 65)

s2_mask = subjects == 2
X_s2    = X[s2_mask]
y_s2    = y[s2_mask]

bm, bs = compute_personal_baseline(X_s2, y_s2)
X_s2_norm = apply_personal_normalization(
    X_s2, bm, bs, feature_cols
)

print(f"NORMALIZED VALUES BY CLASS (S2) — FIXED:")
print(f"Expected: Normal≈baseline, "
      f"Stress further from baseline")
print()
print(f"  {'Feature':20s} {'Normal':10s} "
      f"{'Pre-stress':12s} {'Stress':10s} "
      f"{'Type':8s}")
print("-" * 65)

key_feats = ['AVNN', 'MeanHR', 'SCL_mean',
             'SCR_count', 'SD1', 'EDA_mean',
             'deriv1_std']

for feat in key_feats:
    if feat not in feature_cols:
        continue
    idx  = feature_cols.index(feat)
    ftype = 'EDA' if feat in EDA_FEATURES else 'HRV/ACC'
    vals = []
    for lid in [0, 1, 2]:
        mask = (y_s2 == lid)
        if mask.sum() > 0:
            vals.append(X_s2_norm[mask, idx].mean())
        else:
            vals.append(float('nan'))
    print(f"  {feat:20s} "
          f"{vals[0]:8.3f}   "
          f"{vals[1]:10.3f}   "
          f"{vals[2]:8.3f}   "
          f"{ftype}")

print()
print("EDA features: values near 1.0 = at baseline level")
print("HRV features: values near 0.0 = at baseline level")
print()
print("Cell 2 FIXED complete")

LAYER 1 FIXED — ROBUST NORMALIZATION TEST ON S2
NORMALIZED VALUES BY CLASS (S2) — FIXED:
Expected: Normal≈baseline, Stress further from baseline

  Feature              Normal     Pre-stress   Stress     Type    
-----------------------------------------------------------------
  AVNN                    0.055       -1.135     -0.425   HRV/ACC
  MeanHR                 -0.056        1.271      0.446   HRV/ACC
  SCL_mean                0.322        0.281      0.569   EDA
  SCR_count               1.094        3.000      3.050   EDA
  SD1                    -1.079       -2.260     -1.942   HRV/ACC
  EDA_mean                0.322        0.281      0.569   EDA
  deriv1_std              0.000       -0.000      0.000   EDA

EDA features: values near 1.0 = at baseline level
HRV features: values near 0.0 = at baseline level

Cell 2 FIXED complete


In [29]:
# ============================================================
# CELL 3 — LAYER 2: EMA CONTINUAL LEARNING
#
# What it does:
#   After each session ends:
#   Update personal baseline using EMA formula:
#   new_baseline = α × session_mean + (1-α) × old_baseline
#
#   α = 0.2 (learning rate)
#   Small α = slow adaptation (stable but slow)
#   Large α = fast adaptation (responsive but unstable)
#   0.2 is the standard value from Paper 16
#
# Simulation approach:
#   Since we cannot run real multi-session experiments
#   we simulate by splitting each subject's data into
#   3 equal sessions and updating EMA between sessions
#   This proves the concept works on WESAD data

# Only change: pass feature_cols to normalization function
# ============================================================

class PersonalizedStressModel:

    def __init__(self, base_model, feature_cols,
                 alpha=0.2, n_baseline_windows=3):
        self.base_model         = base_model
        self.feature_cols       = feature_cols
        self.alpha              = alpha
        self.n_baseline_windows = n_baseline_windows
        self.baseline_mean      = None
        self.baseline_std       = None
        self.session_count      = 0

    def calibrate(self, X_session, y_session):
        bm, bs = compute_personal_baseline(
            X_session, y_session,
            self.n_baseline_windows
        )
        if self.baseline_mean is None:
            self.baseline_mean = bm
            self.baseline_std  = bs
        else:
            self.baseline_mean = (
                self.alpha * bm +
                (1 - self.alpha) * self.baseline_mean
            )
            self.baseline_std = (
                self.alpha * bs +
                (1 - self.alpha) * self.baseline_std
            )
        self.session_count += 1

    def predict(self, X_session, scaler):
        # Use FIXED normalization with feature_cols
        X_norm = apply_personal_normalization(
            X_session,
            self.baseline_mean,
            self.baseline_std,
            self.feature_cols       # ← KEY FIX
        )
        X_scaled = scaler.transform(X_norm)
        return self.base_model.predict(X_scaled)


def simulate_sessions(X_person, y_person, subject_id,
                      base_model, scaler, feature_cols,
                      n_sessions=3, alpha=0.2):

    n_windows    = len(X_person)
    session_size = n_windows // n_sessions

    personal_model = PersonalizedStressModel(
        base_model=base_model,
        feature_cols=feature_cols,
        alpha=alpha
    )

    session_results = []

    for session_idx in range(n_sessions):
        start = session_idx * session_size
        end   = start + session_size \
                if session_idx < n_sessions - 1 \
                else n_windows

        X_sess = X_person[start:end]
        y_sess = y_person[start:end]

        if len(X_sess) == 0:
            continue

        personal_model.calibrate(X_sess, y_sess)
        y_pred = personal_model.predict(X_sess, scaler)

        f1_mac = f1_score(y_sess, y_pred,
                          average='macro', zero_division=0)
        f1_pre = f1_score(y_sess, y_pred,
                          labels=[1], average='macro',
                          zero_division=0)
        f1_str = f1_score(y_sess, y_pred,
                          labels=[2], average='macro',
                          zero_division=0)

        session_results.append({
            'session':      session_idx + 1,
            'n_windows':    len(X_sess),
            'n_prestress':  (y_sess == 1).sum(),
            'f1_macro':     f1_mac,
            'f1_prestress': f1_pre,
            'f1_stress':    f1_str
        })

    return session_results


# ── Test on S2 ────────────────────────────────────────────────
print("=" * 65)
print("LAYER 2 FIXED — EMA TEST ON S2")
print("=" * 65)

s2_mask = subjects == 2
X_s2    = X[s2_mask]
y_s2    = y[s2_mask]

results_s2 = simulate_sessions(
    X_s2, y_s2, 2,
    best_model, scaler_obj1,
    feature_cols, n_sessions=3, alpha=0.2
)

print(f"{'Session':8s} {'Windows':8s} {'PS':6s} "
      f"{'F1-macro':10s} {'F1-PreStr':10s} {'F1-Stress':10s}")
print("-" * 55)

for r in results_s2:
    print(f"  {r['session']:6d}   "
          f"{r['n_windows']:6d}   "
          f"{r['n_prestress']:4d}   "
          f"{r['f1_macro']:8.3f}   "
          f"{r['f1_prestress']:8.3f}   "
          f"{r['f1_stress']:8.3f}")

print()
print("Cell 3 FIXED complete")

LAYER 2 FIXED — EMA TEST ON S2
Session  Windows  PS     F1-macro   F1-PreStr  F1-Stress 
-------------------------------------------------------
       1       19      0      1.000      0.000      0.000
       2       19      0      1.000      0.000      0.000
       3       19      2      0.100      0.000      0.300

Cell 3 FIXED complete


In [30]:
# ============================================================
# CELL 4 — LAYER 3: PRE-STRESS TREND DETECTOR
#
# What it does:
#   Monitors last 5 windows for rising stress trend
#   Uses 4 physiological indicators:
#     1. SCL_mean rising   (GSR baseline going up)
#     2. AVNN falling      (heart rate going up)
#     3. SCR_count rising  (more stress responses)
#     4. deriv1_std rising (EDA changing faster)
#
#   If 3 out of 4 indicators show consistent rising/falling
#   AND model currently predicts Normal
#   → Override prediction to Pre-stress (1)
#
# Why these 4 indicators:
#   All selected by feature selection (2+ votes)
#   All show physiological change during pre-stress
#   All available from your ring sensors
# ============================================================

class PreStressTrendDetector:
    """
    Sliding window trend detector for pre-stress early warning.

    Watches last 5 windows for rising stress trends.
    Overrides Normal predictions to Pre-stress when
    3 out of 4 indicators show consistent trending.
    """

    def __init__(self, window_size=5, threshold=3):
        """
        Parameters:
            window_size : number of windows to look back (5)
            threshold   : minimum indicators needed (3 of 4)
        """
        self.window_size = window_size
        self.threshold   = threshold
        self.history     = []

    def update(self, features_dict):
        """
        Add new window features to history.
        features_dict: {feature_name: value}
        """
        self.history.append(features_dict)
        if len(self.history) > self.window_size:
            self.history.pop(0)

    def detect_trend(self):
        """
        Check if last N windows show rising stress trend.

        Returns:
            True  = pre-stress trend detected
            False = no consistent trend
        """

        if len(self.history) < self.window_size:
            return False

        indicators_triggered = 0

        # ── Indicator 1: SCL_mean rising ─────────────────────
        # GSR baseline consistently going up
        if 'SCL_mean' in self.history[0]:
            scl_values = [
                w['SCL_mean'] for w in self.history
            ]
            # Check if each window is higher than previous
            rising = all(
                scl_values[i] > scl_values[i-1]
                for i in range(1, len(scl_values))
            )
            # Also accept mostly rising (4 out of 4 differences)
            diffs  = [
                scl_values[i] - scl_values[i-1]
                for i in range(1, len(scl_values))
            ]
            mostly_rising = sum(d > 0 for d in diffs) >= 3
            if rising or mostly_rising:
                indicators_triggered += 1

        # ── Indicator 2: AVNN falling ─────────────────────────
        # Heart rate increasing (IBI getting shorter)
        if 'AVNN' in self.history[0]:
            avnn_values = [
                w['AVNN'] for w in self.history
            ]
            diffs = [
                avnn_values[i] - avnn_values[i-1]
                for i in range(1, len(avnn_values))
            ]
            # Falling = negative differences
            mostly_falling = sum(d < 0 for d in diffs) >= 3
            if mostly_falling:
                indicators_triggered += 1

        # ── Indicator 3: SCR_count rising ────────────────────
        # More skin conductance responses per minute
        if 'SCR_count' in self.history[0]:
            scr_values = [
                w['SCR_count'] for w in self.history
            ]
            diffs = [
                scr_values[i] - scr_values[i-1]
                for i in range(1, len(scr_values))
            ]
            mostly_rising = sum(d > 0 for d in diffs) >= 3
            if mostly_rising:
                indicators_triggered += 1

        # ── Indicator 4: deriv1_std rising ───────────────────
        # EDA changing more rapidly
        if 'deriv1_std' in self.history[0]:
            d1_values = [
                w['deriv1_std'] for w in self.history
            ]
            diffs = [
                d1_values[i] - d1_values[i-1]
                for i in range(1, len(d1_values))
            ]
            mostly_rising = sum(d > 0 for d in diffs) >= 3
            if mostly_rising:
                indicators_triggered += 1

        return indicators_triggered >= self.threshold

    def apply_override(self, predictions, X_windows):
        """
        Apply trend detection override to model predictions.

        For each window:
          - Update history
          - Check trend
          - If trend detected AND prediction is Normal
            → Override to Pre-stress

        Parameters:
            predictions : model predictions array
            X_windows   : feature matrix for these windows

        Returns:
            overridden  : updated predictions
            n_overrides : count of overrides applied
        """

        overridden  = predictions.copy()
        n_overrides = 0
        self.history = []  # Reset for new session

        for i in range(len(predictions)):

            # Build feature dict for this window
            feat_dict = {
                feature_cols[j]: X_windows[i, j]
                for j in range(len(feature_cols))
            }

            # Update sliding history
            self.update(feat_dict)

            # Check trend and apply override
            if (predictions[i] == 0 and
                    self.detect_trend()):
                overridden[i] = 1  # Override to pre-stress
                n_overrides  += 1

        return overridden, n_overrides


# ── Test trend detector on S2 ─────────────────────────────────
print("=" * 65)
print("LAYER 3 — PRE-STRESS TREND DETECTOR TEST ON S2")
print("=" * 65)

detector = PreStressTrendDetector(
    window_size=5,
    threshold=3
)

# Get S2 pre-stress and surrounding windows
s2_mask = subjects == 2
X_s2    = X[s2_mask]
y_s2    = y[s2_mask]

# Use personal model to predict
personal_model_s2 = PersonalizedStressModel(
    base_model=best_model,
    feature_cols=feature_cols,
    alpha=0.2
)
personal_model_s2.calibrate(X_s2, y_s2)
y_pred_s2 = personal_model_s2.predict(X_s2, scaler_obj1)

# Apply trend detector override
y_overridden, n_overrides = detector.apply_override(
    y_pred_s2, X_s2
)

# Evaluate both
f1_before = f1_score(
    y_s2, y_pred_s2,
    labels=[1], average='macro', zero_division=0
)
f1_after = f1_score(
    y_s2, y_overridden,
    labels=[1], average='macro', zero_division=0
)

print(f"S2 windows: {len(X_s2)}")
print(f"True pre-stress windows: {(y_s2==1).sum()}")
print(f"Overrides applied: {n_overrides}")
print()
print(f"F1-Pre-stress before trend detector: {f1_before:.3f}")
print(f"F1-Pre-stress after  trend detector: {f1_after:.3f}")
print(f"Change: {f1_after - f1_before:+.3f}")
print()
print("Cell 4 complete — trend detector ready")

LAYER 3 — PRE-STRESS TREND DETECTOR TEST ON S2
S2 windows: 57
True pre-stress windows: 2
Overrides applied: 4

F1-Pre-stress before trend detector: 0.000
F1-Pre-stress after  trend detector: 0.000
Change: +0.000

Cell 4 complete — trend detector ready


In [31]:
# ============================================================
# CELL 5 CORRECTED — PROPER PERSONALIZED LOSO
#
# Key fix: retrain model on personally normalized data
# Both training and test normalized before model sees them
#
# Architecture:
#   For each test subject:
#     1. Compute personal baseline for test subject
#     2. Normalize test subject features by their baseline
#     3. For each training subject:
#        compute their personal baseline
#        normalize their features by their baseline
#     4. Train XGBoost on all normalized training data
#     5. Predict on normalized test data
# ============================================================

from sklearn.metrics import confusion_matrix
print("=" * 65)
print("OBJECTIVE 2 CORRECTED — PROPER PERSONALIZED LOSO")
print("=" * 65)

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

unique_subjects = sorted(np.unique(subjects))

results_corrected = {
    'static':     {'true': [], 'pred': []},
    'personal':   {'true': [], 'pred': []},
    'full':       {'true': [], 'pred': []}
}

for sid in unique_subjects:

    print(f"  S{sid}...", end=" ", flush=True)

    sid_mask   = subjects == sid
    X_test     = X[sid_mask]
    y_test     = y[sid_mask]

    train_mask = ~sid_mask
    X_train_all = X[train_mask]
    y_train_all = y[train_mask]
    s_train_all = subjects[train_mask]

    # ── Approach 1: Static (Objective 1 baseline) ────────────
    scaler_s     = StandardScaler()
    X_tr_s       = scaler_s.fit_transform(X_train_all)
    X_te_s       = scaler_s.transform(X_test)

    sw = compute_sample_weight('balanced', y_train_all)
    m_static = xgb.XGBClassifier(
        n_estimators=300, max_depth=6,
        learning_rate=0.1,
        use_label_encoder=False,
        eval_metric='mlogloss',
        random_state=42, n_jobs=-1, verbosity=0
    )
    m_static.fit(X_tr_s, y_train_all, sample_weight=sw)
    y_pred_static = m_static.predict(X_te_s)

    results_corrected['static']['true'].extend(
        y_test.tolist()
    )
    results_corrected['static']['pred'].extend(
        y_pred_static.tolist()
    )

    # ── Approach 2: Personalized — normalize all subjects ────
    # Normalize test subject by their own baseline
    bm_test, bs_test = compute_personal_baseline(
        X_test, y_test
    )
    X_test_norm = apply_personal_normalization(
        X_test, bm_test, bs_test, feature_cols
    )

    # Normalize each training subject by their own baseline
    X_train_norm_list = []
    for train_sid in np.unique(s_train_all):
        mask_tr      = s_train_all == train_sid
        X_tr_subj    = X_train_all[mask_tr]
        y_tr_subj    = y_train_all[mask_tr]

        bm_tr, bs_tr = compute_personal_baseline(
            X_tr_subj, y_tr_subj
        )
        X_tr_norm    = apply_personal_normalization(
            X_tr_subj, bm_tr, bs_tr, feature_cols
        )
        X_train_norm_list.append(X_tr_norm)

    X_train_norm = np.vstack(X_train_norm_list)

    # Scale normalized data
    scaler_p     = StandardScaler()
    X_tr_p       = scaler_p.fit_transform(X_train_norm)
    X_te_p       = scaler_p.transform(X_test_norm)

    # Train on normalized data
    sw_p = compute_sample_weight('balanced', y_train_all)
    m_pers = xgb.XGBClassifier(
        n_estimators=300, max_depth=6,
        learning_rate=0.1,
        use_label_encoder=False,
        eval_metric='mlogloss',
        random_state=42, n_jobs=-1, verbosity=0
    )
    m_pers.fit(X_tr_p, y_train_all, sample_weight=sw_p)
    y_pred_pers = m_pers.predict(X_te_p)

    results_corrected['personal']['true'].extend(
        y_test.tolist()
    )
    results_corrected['personal']['pred'].extend(
        y_pred_pers.tolist()
    )

    # ── Approach 3: Full system + trend detector ──────────────
    detector = PreStressTrendDetector(
        window_size=5, threshold=3
    )
    y_pred_full, _ = detector.apply_override(
        y_pred_pers, X_test_norm
    )

    results_corrected['full']['true'].extend(
        y_test.tolist()
    )
    results_corrected['full']['pred'].extend(
        y_pred_full.tolist()
    )

    print(f"done")

# ── Print results ─────────────────────────────────────────────
print()
print("=" * 75)
print("CORRECTED OBJECTIVE 2 — RESULTS")
print("=" * 75)
print(f"{'Approach':30s} {'F1-macro':10s} "
      f"{'F1-PreStr':10s} {'F1-Stress':10s} "
      f"{'OrdMAE':8s}")
print("-" * 75)

base_f1_pre = None
for name, key in [
    ('Static XGBoost (Obj1)',     'static'),
    ('Personalized baseline',     'personal'),
    ('+ Trend detector (Full)',   'full')
]:
    t = np.array(results_corrected[key]['true'])
    p = np.array(results_corrected[key]['pred'])

    f1_mac = f1_score(t, p, average='macro',
                      zero_division=0)
    f1_pre = f1_score(t, p, labels=[1],
                      average='macro', zero_division=0)
    f1_str = f1_score(t, p, labels=[2],
                      average='macro', zero_division=0)
    mae    = np.mean(np.abs(t - p))

    if base_f1_pre is None:
        base_f1_pre = f1_pre
        change_str  = ""
    else:
        change = f1_pre - base_f1_pre
        change_str = f" ({change:+.3f})"

    print(f"  {name:28s} "
          f"{f1_mac:8.3f}   "
          f"{f1_pre:8.3f}{change_str:10s} "
          f"{f1_str:8.3f}   "
          f"{mae:6.3f}")

# ── Full report ───────────────────────────────────────────────
t = np.array(results_corrected['full']['true'])
p = np.array(results_corrected['full']['pred'])

total_imp = (
    f1_score(t, p, labels=[1],
             average='macro', zero_division=0) -
    base_f1_pre
)

print()
print(f"Total F1-Pre-stress improvement: {total_imp:+.3f}")
print(f"Target >5%: "
      f"{'MET ✓' if total_imp >= 0.05 else 'not met'}")

print()
print("FULL SYSTEM REPORT:")
print(classification_report(
    t, p,
    target_names=['Normal', 'Pre-stress', 'Stress'],
    digits=3
))

print()
print("CONFUSION MATRIX:")
cm = confusion_matrix(t, p)
print(f"                 Normal  Pre-str  Stress")
for i, row in enumerate(cm):
    lbl = ['Normal','Pre-str','Stress'][i]
    print(f"  Actual {lbl:7s}: "
          f"{row[0]:6d}   {row[1]:6d}   {row[2]:6d}")

OBJECTIVE 2 CORRECTED — PROPER PERSONALIZED LOSO
  S2... 

done
  S3... done
  S4... done
  S5... done
  S6... done
  S7... done
  S8... done
  S9... done
  S10... done
  S11... done
  S13... done
  S14... done
  S15... done
  S16... done
  S17... done

CORRECTED OBJECTIVE 2 — RESULTS
Approach                       F1-macro   F1-PreStr  F1-Stress  OrdMAE  
---------------------------------------------------------------------------
  Static XGBoost (Obj1)           0.617      0.235              0.706    0.296
  Personalized baseline           0.634      0.150 (-0.085)     0.802    0.179
  + Trend detector (Full)         0.628      0.144 (-0.091)     0.802    0.194

Total F1-Pre-stress improvement: -0.091
Target >5%: not met

FULL SYSTEM REPORT:
              precision    recall  f1-score   support

      Normal      0.943     0.934     0.938       587
  Pre-stress      0.103     0.241     0.144        29
      Stress      0.859     0.753     0.802       267

    accuracy                          0.856       883
   macro avg      0.635     0.643

In [32]:
# ============================================================
# CELL 6 — SAVE OBJECTIVE 2 MODEL AND RESULTS
# ============================================================

print("=" * 65)
print("SAVING OBJECTIVE 2 RESULTS")
print("=" * 65)

# Save comparison results
rows = []
for name, key in approaches:
    t = np.array(results[key]['true'])
    p = np.array(results[key]['pred'])
    rows.append({
        'approach':     name,
        'f1_macro':     f1_score(t, p, average='macro',
                                 zero_division=0),
        'f1_prestress': f1_score(t, p, labels=[1],
                                 average='macro',
                                 zero_division=0),
        'f1_stress':    f1_score(t, p, labels=[2],
                                 average='macro',
                                 zero_division=0),
        'ordinal_mae':  np.mean(np.abs(t - p))
    })

res_df = pd.DataFrame(rows)
save_path = os.path.join(
    RESULTS_PATH, "objective2",
    "obj2_comparison.csv"
)
os.makedirs(
    os.path.join(RESULTS_PATH, "objective2"),
    exist_ok=True
)
res_df.to_csv(save_path, index=False)
print(f"Results saved: {save_path}")

# Save full system components
models_path = os.path.join(COMPONENT_PATH, "saved_models")

# Save the EMA model class and detector
joblib.dump(
    PersonalizedStressModel,
    os.path.join(models_path, "personal_model_class.pkl")
)
joblib.dump(
    PreStressTrendDetector,
    os.path.join(models_path, "trend_detector_class.pkl")
)

print(f"Model classes saved")
print()
print("=" * 65)
print("OBJECTIVE 2 COMPLETE")
print("=" * 65)
print()
print("Next steps:")
print("  1. Review results table")
print("  2. If improvement > 5% → target met")
print("  3. Build Flask API for demo")
print("  4. Build mobile app for viva")
print("  5. Collect real ring data (Objective 3)")

SAVING OBJECTIVE 2 RESULTS
Results saved: C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals\results\objective2\obj2_comparison.csv
Model classes saved

OBJECTIVE 2 COMPLETE

Next steps:
  1. Review results table
  2. If improvement > 5% → target met
  3. Build Flask API for demo
  4. Build mobile app for viva
  5. Collect real ring data (Objective 3)


In [35]:
# ============================================================
# CELL 7 COMPLETE — SWELL CROSS-DATASET VALIDATION
#
# SWELL column → WESAD feature mapping:
#   MEAN_RR → AVNN     (mean RR interval)
#   HR      → MeanHR   (mean heart rate)
#   SDRR    → SDHR     (std of RR intervals)
#   pNN25   → pNN20    (closest available, documented)
#   SD1     → SD1      (exact match)
#
# SWELL condition → stress class mapping:
#   no stress    → Normal (0)
#   time pressure→ Pre-stress (1)
#   interruption → Stress (2)
# ============================================================

import pandas as pd
import numpy as np
import os
import sys
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, classification_report,
    confusion_matrix
)
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb

print("=" * 65)
print("SWELL CROSS-DATASET VALIDATION")
print("=" * 65)

# ── Step 1: Load SWELL ────────────────────────────────────────
SWELL_PATH = r"C:\Users\weha1\datasets\SWELL\archive\hrv dataset\data\final"

swell_train = pd.read_csv(
    os.path.join(SWELL_PATH, "train.csv")
)
swell_test  = pd.read_csv(
    os.path.join(SWELL_PATH, "test.csv")
)
swell_df = pd.concat(
    [swell_train, swell_test],
    ignore_index=True
)

print(f"SWELL loaded: {swell_df.shape}")

# ── Step 2: Map conditions to labels ─────────────────────────
print("\nStep 2: Mapping conditions to stress classes")

condition_map = {
    'no stress':    0,  # Normal
    'time pressure': 1,  # Pre-stress
    'interruption': 2   # Stress
}

swell_df['label'] = swell_df['condition'].map(condition_map)

# Drop any unmapped rows
swell_df = swell_df.dropna(subset=['label'])
swell_df['label'] = swell_df['label'].astype(int)

print(f"  Label distribution after mapping:")
for lid, name in {
    0:'Normal', 1:'Pre-stress', 2:'Stress'
}.items():
    count = (swell_df['label'] == lid).sum()
    pct   = count / len(swell_df) * 100
    print(f"    Class {lid} ({name:10s}): "
          f"{count:7d} ({pct:.1f}%)")

# ── Step 3: Extract matching HRV features ────────────────────
print("\nStep 3: Extracting matching HRV features")

# Map SWELL columns to your feature names
swell_feature_map = {
    'MEAN_RR': 'AVNN',    # Mean RR interval (ms)
    'HR':      'MeanHR',  # Mean heart rate (bpm)
    'SDRR':    'SDHR',    # Std of RR intervals
    'pNN25':   'pNN20',   # Closest to pNN20 (documented)
    'SD1':     'SD1'      # Exact match
}

# Rename columns to match your feature names
swell_renamed = swell_df.rename(
    columns=swell_feature_map
)

# Features available in both datasets
common_features = ['AVNN', 'MeanHR', 'SDHR', 'pNN20', 'SD1']

print(f"  Common features (HRV only): {common_features}")
print(f"  Note: pNN25 used as substitute for pNN20")
print(f"  Note: GSR features not available in SWELL")
print(f"        This is a PARTIAL cross-validation")
print(f"        HRV features only (5 of 22)")

# ── Step 4: Clean SWELL data ──────────────────────────────────
print("\nStep 4: Cleaning SWELL data")

X_swell = swell_renamed[common_features].values
y_swell = swell_renamed['label'].values

# Check for NaN
nan_count = np.isnan(X_swell).sum()
print(f"  NaN values: {nan_count}")

if nan_count > 0:
    from sklearn.impute import SimpleImputer
    imputer  = SimpleImputer(strategy='median')
    X_swell  = imputer.fit_transform(X_swell)
    print(f"  NaN filled with median")

# Check for infinite values
inf_count = np.isinf(X_swell).sum()
print(f"  Infinite values: {inf_count}")

if inf_count > 0:
    X_swell = np.where(
        np.isinf(X_swell),
        np.nan,
        X_swell
    )
    from sklearn.impute import SimpleImputer
    imputer = SimpleImputer(strategy='median')
    X_swell = imputer.fit_transform(X_swell)
    print(f"  Infinite values replaced")

print(f"  Clean SWELL shape: {X_swell.shape}")

# ── Step 5: Prepare WESAD training data ──────────────────────
print("\nStep 5: Preparing WESAD training data")
print("  Loading wesad_features_selected.csv...")

COMPONENT_PATH = r"C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals"
sys.path.insert(0, os.path.join(COMPONENT_PATH, "src"))
from utils.config import RESULTS_PATH

csv_path    = os.path.join(
    RESULTS_PATH, "objective1",
    "wesad_features_selected.csv"
)
features_df  = pd.read_csv(csv_path)
feature_cols = [c for c in features_df.columns
                if c not in ['label', 'subject_id']]

# Extract only common HRV features from WESAD
X_wesad_full = features_df[feature_cols].values
y_wesad      = features_df['label'].values

# Get indices of common features in WESAD
wesad_common_idx = [
    feature_cols.index(f)
    for f in common_features
    if f in feature_cols
]
available_features = [
    feature_cols[i] for i in wesad_common_idx
]

X_wesad = X_wesad_full[:, wesad_common_idx]

print(f"  WESAD features available: {available_features}")
print(f"  WESAD training shape: {X_wesad.shape}")
print(f"  WESAD label distribution:")
for lid, name in {
    0:'Normal', 1:'Pre-stress', 2:'Stress'
}.items():
    count = (y_wesad == lid).sum()
    print(f"    Class {lid} ({name:10s}): {count}")

# ── Step 6: Train on WESAD, test on SWELL ────────────────────
print("\nStep 6: Training on WESAD, testing on SWELL")
print("  No SWELL data used in training")
print("  Pure cross-dataset generalization test")

# Scale WESAD training data
scaler_cross = StandardScaler()
X_wesad_scaled = scaler_cross.fit_transform(X_wesad)

# Scale SWELL test data with WESAD scaler
# Important: use WESAD statistics not SWELL statistics
# This tests true generalization
X_swell_scaled = scaler_cross.transform(X_swell)

# Train XGBoost on all WESAD data
sample_weights = compute_sample_weight(
    'balanced', y_wesad
)

xgb_cross = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb_cross.fit(
    X_wesad_scaled,
    y_wesad,
    sample_weight=sample_weights
)

print(f"  Model trained on WESAD ✓")

# Predict on SWELL
y_pred_swell = xgb_cross.predict(X_swell_scaled)
print(f"  Predictions made on SWELL ✓")

# ── Step 7: Evaluate results ──────────────────────────────────
print("\n" + "=" * 65)
print("SWELL CROSS-DATASET VALIDATION RESULTS")
print("=" * 65)

f1_mac = f1_score(
    y_swell, y_pred_swell,
    average='macro', zero_division=0
)
f1_pre = f1_score(
    y_swell, y_pred_swell,
    labels=[1], average='macro', zero_division=0
)
f1_nor = f1_score(
    y_swell, y_pred_swell,
    labels=[0], average='macro', zero_division=0
)
f1_str = f1_score(
    y_swell, y_pred_swell,
    labels=[2], average='macro', zero_division=0
)
mae    = np.mean(np.abs(y_swell - y_pred_swell))
acc    = np.mean(y_swell == y_pred_swell)

print(f"\nMetric          Value")
print(f"─────────────────────────")
print(f"Accuracy:       {acc:.3f}")
print(f"F1-macro:       {f1_mac:.3f}")
print(f"F1-Normal:      {f1_nor:.3f}")
print(f"F1-Pre-stress:  {f1_pre:.3f}")
print(f"F1-Stress:      {f1_str:.3f}")
print(f"Ordinal MAE:    {mae:.3f}")

print(f"\nComparison:")
print(f"  {'Metric':20s} {'WESAD LOSO':12s} {'SWELL Cross':12s}")
print(f"  {'─'*45}")
print(f"  {'F1-macro':20s} {'0.617':12s} {f1_mac:.3f}")
print(f"  {'F1-Pre-stress':20s} {'0.286':12s} {f1_pre:.3f}")
print(f"  {'F1-Stress':20s} {'0.689':12s} {f1_str:.3f}")
print(f"  {'Ordinal MAE':20s} {'0.357':12s} {mae:.3f}")

print(f"\nDetailed classification report:")
print(classification_report(
    y_swell,
    y_pred_swell,
    target_names=['Normal', 'Pre-stress', 'Stress'],
    digits=3
))

print(f"Confusion matrix:")
cm = confusion_matrix(y_swell, y_pred_swell)
print(f"                 Normal   Pre-str   Stress")
for i, row in enumerate(cm):
    lbl = ['Normal', 'Pre-str', 'Stress'][i]
    print(f"  Actual {lbl:7s}: "
          f"{row[0]:8d}  {row[1]:8d}  {row[2]:8d}")

# ── Step 8: Interpretation ────────────────────────────────────
print(f"\n{'='*65}")
print(f"INTERPRETATION")
print(f"{'='*65}")

print(f"""
Important notes for paper documentation:

1. Feature limitation:
   Only 5 HRV features used (of 22 total)
   SWELL has no GSR/EDA signals
   This is a PARTIAL cross-validation
   Full validation requires GSR — only possible
   with Objective 3 real ring data

2. Dataset difference:
   WESAD: controlled lab stress (TSST)
   SWELL: real office work stress
   Different stress context and magnitude
   Some performance drop expected and acceptable

3. What good results mean:
   HRV patterns from WESAD generalize to
   office stress context in SWELL
   Core physiological stress response is consistent
   across different stress induction methods

4. What poor results mean:
   HRV features alone insufficient for cross-dataset
   GSR features (68% of your model) are needed
   Full 22-feature model needed for good generalization
   This motivates real ring deployment
""")

# ── Step 9: Save results ──────────────────────────────────────
results_path = os.path.join(
    RESULTS_PATH, "objective2",
    "swell_cross_validation.csv"
)
os.makedirs(
    os.path.join(RESULTS_PATH, "objective2"),
    exist_ok=True
)

swell_results = pd.DataFrame([{
    'dataset':      'SWELL',
    'features':     'HRV only (5 of 22)',
    'train_on':     'WESAD all 15 subjects',
    'test_on':      'SWELL 25 subjects',
    'accuracy':     acc,
    'f1_macro':     f1_mac,
    'f1_normal':    f1_nor,
    'f1_prestress': f1_pre,
    'f1_stress':    f1_str,
    'ordinal_mae':  mae,
    'note':         'pNN25 used as pNN20 substitute'
}])

swell_results.to_csv(results_path, index=False)
print(f"Results saved: {results_path}")
print(f"\nSWELL CROSS-VALIDATION COMPLETE")
print(f"Next step: Flask API backend")

SWELL CROSS-DATASET VALIDATION
SWELL loaded: (410322, 36)

Step 2: Mapping conditions to stress classes
  Label distribution after mapping:
    Class 0 (Normal    ):  222240 (54.2%)
    Class 1 (Pre-stress):   71150 (17.3%)
    Class 2 (Stress    ):  116932 (28.5%)

Step 3: Extracting matching HRV features
  Common features (HRV only): ['AVNN', 'MeanHR', 'SDHR', 'pNN20', 'SD1']
  Note: pNN25 used as substitute for pNN20
  Note: GSR features not available in SWELL
        This is a PARTIAL cross-validation
        HRV features only (5 of 22)

Step 4: Cleaning SWELL data
  NaN values: 0
  Infinite values: 0
  Clean SWELL shape: (410322, 5)

Step 5: Preparing WESAD training data
  Loading wesad_features_selected.csv...
  WESAD features available: ['AVNN', 'MeanHR', 'SDHR', 'pNN20', 'SD1']
  WESAD training shape: (883, 5)
  WESAD label distribution:
    Class 0 (Normal    ): 587
    Class 1 (Pre-stress): 29
    Class 2 (Stress    ): 267

Step 6: Training on WESAD, testing on SWELL
  No SWE

In [37]:
# ============================================================
# MODEL VERIFICATION — confirm saved files exist
# Run this before building Flask backend
# ============================================================

import joblib
import os
import numpy as np

COMPONENT_PATH = r"C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals"
models_path    = os.path.join(COMPONENT_PATH, "saved_models")

print("=" * 55)
print("CHECKING SAVED MODEL FILES")
print("=" * 55)

files_to_check = [
    "best_model_obj1.pkl",
    "scaler_obj1.pkl",
    "feature_cols_obj1.pkl"
]

all_good = True

for fname in files_to_check:
    fpath  = os.path.join(models_path, fname)
    exists = os.path.exists(fpath)
    size   = os.path.getsize(fpath)/1024 if exists else 0
    status = "FOUND ✓" if exists else "MISSING ✗"
    print(f"  {fname:30s} {status}  ({size:.1f} KB)")
    if not exists:
        all_good = False

print()

if all_good:
    print("Loading model...")
    model     = joblib.load(
        os.path.join(models_path, "best_model_obj1.pkl")
    )
    scaler    = joblib.load(
        os.path.join(models_path, "scaler_obj1.pkl")
    )
    feat_cols = joblib.load(
        os.path.join(models_path, "feature_cols_obj1.pkl")
    )
    print(f"  Model type:    {type(model).__name__}")
    print(f"  Features:      {len(feat_cols)}")
    print(f"  Feature names: {feat_cols}")
    print()

    # Quick test prediction
    print("Running test prediction...")
    test_input  = np.zeros((1, len(feat_cols)))
    test_scaled = scaler.transform(test_input)
    test_pred   = model.predict(test_scaled)
    test_proba  = model.predict_proba(test_scaled)
    print(f"  Test prediction:    {test_pred[0]}")
    print(f"  Test probabilities: {test_proba[0].round(3)}")
    print()
    print("ALL FILES VERIFIED ✓")
    print("Ready to build Flask backend")
else:
    print("MISSING FILES — need to rerun Cell 6")
    print("of notebook 05_model_training.ipynb")

CHECKING SAVED MODEL FILES
  best_model_obj1.pkl            FOUND ✓  (965.1 KB)
  scaler_obj1.pkl                FOUND ✓  (1.1 KB)
  feature_cols_obj1.pkl          FOUND ✓  (0.3 KB)

Loading model...
  Model type:    XGBClassifier
  Features:      22
  Feature names: ['AVNN', 'MeanHR', 'wavelet_d3_energy', 'SDHR', 'SCR_count', 'SCR_width_mean', 'EDA_q75', 'SCR_amp_mean', 'EDA_mean', 'SCL_mean', 'ACC_mean', 'ACC_std', 'EDA_slope', 'deriv1_std', 'deriv2_mean', 'pNN20', 'EDA_q10', 'EDA_q90', 'EDA_std', 'wavelet_d1_energy', 'EDA_range', 'SD1']

Running test prediction...
  Test prediction:    0
  Test probabilities: [0.974 0.001 0.025]

ALL FILES VERIFIED ✓
Ready to build Flask backend


In [1]:
# Add as a cell in any notebook
import pandas as pd
import os

df = pd.read_csv(
    r"C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals\results\objective1\wesad_features_selected.csv"
)

print("Pre-stress windows per subject:")
ps = df[df['label']==1].groupby('subject_id').size()
print(ps.sort_values(ascending=False))

print("\nStress windows per subject:")
st = df[df['label']==2].groupby('subject_id').size()
print(st.sort_values(ascending=False))

print("\nBest demo subject (most pre-stress):")
print(f"S{ps.idxmax()} with {ps.max()} pre-stress windows")

Pre-stress windows per subject:
subject_id
2     2
3     2
4     2
5     2
6     2
7     2
8     2
9     2
10    2
11    2
13    2
15    2
16    2
17    2
14    1
dtype: int64

Stress windows per subject:
subject_id
10    20
8     19
11    19
15    19
3     18
6     18
13    18
17    18
2     17
4     17
5     17
7     17
9     17
14    17
16    16
dtype: int64

Best demo subject (most pre-stress):
S2 with 2 pre-stress windows


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r"C:\Users\weha1\OneDrive\Documents\ResearchCode"
    r"\ai-neuro-monitor\physiological_signals\results"
    r"\objective1\wesad_features_selected.csv"
)

s10 = df[df['subject_id'] == 10].copy()
print("S10 label distribution:")
print(s10['label'].value_counts().sort_index())
print(f"\nTotal S10 windows: {len(s10)}")

print("\nS10 key feature means by label:")
for lbl, name in [(0,'Normal'),(1,'Pre-stress'),(2,'Stress')]:
    subset = s10[s10['label'] == lbl]
    if len(subset) > 0:
        print(f"\n{name} (n={len(subset)}):")
        print(f"  AVNN:     {subset['AVNN'].mean():.1f} ms")
        print(f"  MeanHR:   {subset['MeanHR'].mean():.1f} bpm")
        print(f"  SCL_mean: {subset['SCL_mean'].mean():.3f} μS")
        print(f"  SCR_count:{subset['SCR_count'].mean():.2f}")

S10 label distribution:
label
0    39
1     2
2    20
Name: count, dtype: int64

Total S10 windows: 61

S10 key feature means by label:

Normal (n=39):
  AVNN:     606.9 ms
  MeanHR:   99.0 bpm
  SCL_mean: 0.744 μS
  SCR_count:0.49

Pre-stress (n=2):
  AVNN:     474.8 ms
  MeanHR:   126.4 bpm
  SCL_mean: 0.934 μS
  SCR_count:3.00

Stress (n=20):
  AVNN:     570.8 ms
  MeanHR:   105.4 bpm
  SCL_mean: 1.096 μS
  SCR_count:7.45


In [2]:
print("Best subjects for demo — ranked by reliability:\n")
print(f"{'Subject':10s} {'Normal HR':12s} {'Stress HR':12s} {'HR rise':10s} {'SCL rise':10s} {'Score':8s}")
print("-" * 65)

for sid in sorted(df['subject_id'].unique()):
    s = df[df['subject_id'] == sid]
    n  = s[s['label']==0]
    ps = s[s['label']==1]
    st = s[s['label']==2]
    if len(n)==0 or len(st)==0: continue
    
    hr_normal = n['MeanHR'].mean()
    hr_stress = st['MeanHR'].mean()
    hr_rise   = hr_stress - hr_normal
    
    scl_normal = n['SCL_mean'].mean()
    scl_stress = st['SCL_mean'].mean()
    scl_rise   = scl_stress - scl_normal
    scl_pct    = (scl_rise / scl_normal) * 100 if scl_normal > 0 else 0
    
    # Lower resting HR = more normal physiology
    # Higher HR rise = clearer stress signal
    # Higher SCL rise = clearer GSR signal
    hr_normal_score = max(0, (85 - hr_normal) / 20 * 30)
    hr_rise_score   = min(30, hr_rise / 20 * 30)
    scl_score       = min(40, scl_pct / 100 * 40)
    score           = hr_normal_score + hr_rise_score + scl_score
    
    print(f"S{sid:<9d} {hr_normal:8.1f}bpm  {hr_stress:8.1f}bpm  "
          f"{hr_rise:+7.1f}    {scl_pct:+6.1f}%    {score:6.1f}")

print("\nHigher score = better demo subject")
print("Look for: normal HR near 70, clear HR and SCL rise")

Best subjects for demo — ranked by reliability:

Subject    Normal HR    Stress HR    HR rise    SCL rise   Score   
-----------------------------------------------------------------
S2             72.0bpm      75.6bpm     +3.6     +76.6%      55.6
S3             54.3bpm      82.1bpm    +27.9    +147.2%     116.1
S4             60.7bpm      80.6bpm    +19.8     +86.4%     100.8
S5             62.9bpm      87.7bpm    +24.8    +126.3%     103.1
S6             69.2bpm      77.5bpm     +8.2    +121.6%      76.0
S7             70.6bpm      83.8bpm    +13.2     +82.9%      74.6
S8             70.9bpm      90.1bpm    +19.2     +36.6%      64.7
S9             77.3bpm      80.6bpm     +3.3    +217.8%      56.6
S10            99.0bpm     105.4bpm     +6.4     +47.3%      28.6
S11            74.3bpm     121.6bpm    +47.4    +143.8%      86.1
S13            87.0bpm     107.0bpm    +20.0     +48.6%      49.5
S14            70.5bpm     111.4bpm    +40.9     +20.4%      59.9
S15            82.8bpm   